# Notebook 1 — One-Turn LLM Patterns

In this notebook we'll build the foundations of LLM-powered features for our nanny agency:

1. **Email generation** with prompts — zero-shot → role-conditioned → few-shot → structured output
2. **Personalized birthday wishes** — system/user split, personalization variables
3. **PDF extraction** — turn unstructured resumes/intakes into validated Pydantic records
4. **Embeddings + matching** — vector search in ChromaDB with a 2D UMAP visualization

Total time: ~1h 15m. Each section ends with a **Try it** cell — feel free to tweak prompts, models, and parameters. The on-disk cache (`CachedOpenAI`) means re-runs cost nothing.

**Outputs of this notebook:** the `nanny_db/` Chroma collection, used by Notebook 2.

## 0. Setup & env check

Loads the OpenAI key, sets the repo root so imports work whether you launched Jupyter from the repo root or from `notebooks/`, and runs a one-call smoke test.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# When jupyter is launched from the repo root, the cwd here is notebooks/.
# Add the repo root so we can import the nanny_workshop package and baml_client.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")
key = os.getenv("OPENAI_API_KEY")
assert key and key.startswith("sk-"), "Set OPENAI_API_KEY in .env (see README step 3)."

from nanny_workshop.openai_client import CachedOpenAI

# Disk cache lets you re-run this notebook for free.
client = CachedOpenAI(cache_dir=ROOT / ".cache" / "n1")

# Smoke test: one round trip to OpenAI.
reply = client.complete(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "Reply with one word: ready"}],
)
print(f"Setup OK — model reply: {reply!r}")

## 1. Email generation with prompts

The same task — drafting a booking confirmation email — produces wildly different output depending on how you prompt for it. We'll walk through four progressively-constrained prompt styles and see how output quality and shape change.

The scenario: the agency needs to confirm a booking with a parent. Let's see what the LLM does with increasing amounts of guidance.

In [ ]:
# 1a. ZERO-SHOT: give the model the bare task and see what it does.
# Notice: no role, no tone guidance, no format. We're trusting the model's defaults.

zero_shot_prompt = "Write an email confirming a nanny booking for next Thursday."

zero_shot = client.complete(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": zero_shot_prompt}],
)
print(zero_shot)

In [ ]:
# 1b. ROLE + STYLE: tell the model who it is and how to sound.
# Setting persona ("warm, professional") makes outputs much more consistent across runs.

role_styled = client.complete(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": (
                "You are the Nanny Agency's customer-care assistant. "
                "Tone: warm, professional, reassuring. "
                "Keep emails to 4-6 sentences. Sign as 'The Nanny Agency Team'."
            ),
        },
        {
            "role": "user",
            "content": "Write a booking confirmation email for the Johnson family with nanny Maria on Thursday for 6 hours.",
        },
    ],
)
print(role_styled)

In [ ]:
# 1c. FEW-SHOT: show the model 2 example emails first.
# This teaches output STRUCTURE by demonstration, which is often clearer than instructions.

few_shot_examples = [
    {
        "role": "user",
        "content": "Confirm booking — family: Chen, nanny: Aisha, day: Tuesday, hours: 4.",
    },
    {
        "role": "assistant",
        "content": (
            "Subject: Your Tuesday booking is confirmed\n\n"
            "Hi Chen family,\n\n"
            "We're delighted to confirm Aisha will be with you this Tuesday for 4 hours. "
            "She'll arrive 10 minutes before the start time. Please reply if anything changes.\n\n"
            "Warmly,\nThe Nanny Agency Team"
        ),
    },
    {
        "role": "user",
        "content": "Confirm booking — family: Patel, nanny: Sam, day: Saturday, hours: 5.",
    },
    {
        "role": "assistant",
        "content": (
            "Subject: Saturday with Sam — confirmed\n\n"
            "Hi Patel family,\n\n"
            "All set: Sam will be with you Saturday for 5 hours. "
            "Looking forward to a great session — reach out if anything comes up.\n\n"
            "Warmly,\nThe Nanny Agency Team"
        ),
    },
]

few_shot = client.complete(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You write booking confirmation emails in the format shown."},
        *few_shot_examples,
        {"role": "user", "content": "Confirm booking — family: Johnson, nanny: Maria, day: Thursday, hours: 6."},
    ],
)
print(few_shot)

In [ ]:
# 1d. STRUCTURED OUTPUT: ask for JSON so a downstream system can parse it.
# Use response_format={"type": "json_object"} so the model is constrained to valid JSON.

import json

structured = client.complete(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": (
                "You return ONLY a JSON object with fields: subject (string), greeting (string), "
                "body (string, 2-4 sentences), signature (string). No prose outside JSON."
            ),
        },
        {
            "role": "user",
            "content": "Confirm Johnson family booking with Maria, Thursday, 6 hours.",
        },
    ],
    response_format={"type": "json_object"},
)

parsed = json.loads(structured)
print(json.dumps(parsed, indent=2))
print()
print(f"Type: {type(parsed).__name__}, keys: {list(parsed.keys())}")

In [ ]:
# 🎯 TRY IT: change ONE thing and observe.
#   - Change the tone in the system prompt to "concise and matter-of-fact"
#   - Or swap gpt-4o-mini for gpt-4o
#   - Or set temperature to 1.5 (vs default 0.0) — note temperature is a CachedOpenAI arg
#
# Each different (model, messages, temperature) combo creates a new cache entry,
# so you can compare without re-paying for earlier runs.

your_system = "You are the Nanny Agency's customer-care assistant. Tone: warm, professional, reassuring."
your_user = "Write a booking confirmation email for the Johnson family with nanny Maria on Thursday for 6 hours."

experiment = client.complete(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": your_system},
        {"role": "user", "content": your_user},
    ],
    temperature=0.0,
)
print(experiment)

## 2. Personalized birthday wishes

Two things change in this section compared to section 1:

1. **Personalization variables** — the same prompt template, parameterized per child. Same model, but the *system* prompt holds the constant style guide and the *user* prompt holds the variables.
2. **System vs user split** — the model treats system instructions as more durable; we'll see why this matters when we swap user content across calls.

In [ ]:
# 2a. Define a system prompt with the constant style guide,
# and a user-prompt template with placeholders for personalization variables.

BIRTHDAY_SYSTEM = (
    "You write warm, 2-sentence birthday wishes for children. "
    "Be age-appropriate and personal. Mention at least one of the child's interests."
)

def birthday_wish(child_name: str, age: int, interests: list[str]) -> str:
    interests_str = ", ".join(interests)
    return client.complete(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": BIRTHDAY_SYSTEM},
            {
                "role": "user",
                "content": f"Child name: {child_name}\nAge turning: {age}\nInterests: {interests_str}",
            },
        ],
    )

# Generate a wish for one child.
print(birthday_wish("Mia", 4, ["dinosaurs", "painting"]))

In [ ]:
# 2b. Run the same template across 3 different child profiles.
# Notice the system prompt is unchanged — only the user content varies.

children = [
    {"name": "Mia",   "age": 4, "interests": ["dinosaurs", "painting"]},
    {"name": "Theo",  "age": 7, "interests": ["soccer", "Minecraft", "magic tricks"]},
    {"name": "Aanya", "age": 2, "interests": ["bubbles", "trucks"]},
]

for c in children:
    print(f"--- {c['name']} ({c['age']}) ---")
    print(birthday_wish(c["name"], c["age"], c["interests"]))
    print()

In [ ]:
# 🎯 TRY IT:
#   - Add a `language` parameter and a "Reply in {language}." line to the user prompt.
#   - Generate the same wish in English, Spanish, and Mandarin.
#   - Notice: the system prompt didn't need to change — the system/user split keeps
#     constants and variables on different layers.

def birthday_wish_multilingual(child_name: str, age: int, interests: list[str], language: str = "English") -> str:
    interests_str = ", ".join(interests)
    return client.complete(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": BIRTHDAY_SYSTEM},
            {
                "role": "user",
                "content": (
                    f"Child name: {child_name}\nAge turning: {age}\n"
                    f"Interests: {interests_str}\nReply in {language}."
                ),
            },
        ],
    )

for lang in ["English", "Spanish", "Mandarin"]:
    print(f"--- {lang} ---")
    print(birthday_wish_multilingual("Mia", 4, ["dinosaurs", "painting"], language=lang))
    print()

## 3. PDF extraction — from text to validated records

Real-world LLM features start with messy input: PDFs, scans, screenshots. This section turns 5 nanny resumes and 5 parent intake forms (in `data/pdfs/`) into validated `NannyProfile` and `ParentIntake` Pydantic records.

We'll compare:
1. **Free-form extraction** — "tell me about this resume" → narrative prose
2. **Schema-locked extraction** — JSON matching a Pydantic model → typed records the rest of the app can rely on

Schema-locking is what turns LLMs from a demo into a reliable component.

In [ ]:
# 3a. Load one nanny resume and one parent intake.
from nanny_workshop.pdf_loader import extract_text

PDF_DIR = ROOT / "data" / "pdfs"
nanny_pdf = PDF_DIR / "nanny_resume_01.pdf"
parent_pdf = PDF_DIR / "parent_intake_01.pdf"

nanny_text = extract_text(nanny_pdf)
parent_text = extract_text(parent_pdf)

print("--- NANNY RESUME (first 400 chars) ---")
print(nanny_text[:400])
print()
print("--- PARENT INTAKE (first 400 chars) ---")
print(parent_text[:400])

In [ ]:
# 3b. FREE-FORM: just ask the model to summarize.
# This is fast and looks impressive in a demo. But the output is prose — you can't
# program against it. Try parsing "certifications" out of this reliably; you can't.

freeform = client.complete(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": f"Summarize this nanny resume:\n\n{nanny_text}"},
    ],
)
print(freeform)

In [ ]:
# 3c. SCHEMA-LOCKED: hand the model a Pydantic schema and demand JSON.
# We use the shared prompt template from nanny_workshop/prompts.py and
# validate the response with Pydantic — so bad output crashes here, not later.

import json
from nanny_workshop.models import NannyProfile, ParentIntake
from nanny_workshop.prompts import NANNY_RESUME_EXTRACTION_PROMPT, PARENT_INTAKE_EXTRACTION_PROMPT

raw_json = client.complete(
    model="gpt-4o-mini",
    messages=[{
        "role": "user",
        "content": NANNY_RESUME_EXTRACTION_PROMPT.replace("{resume_text}", nanny_text),
    }],
    temperature=0.0,
    response_format={"type": "json_object"},
)
data = json.loads(raw_json)
data["id"] = "n_demo"

# Pydantic does the real work — bad days, missing fields, out-of-range values all raise.
nanny = NannyProfile.model_validate(data)

print(nanny.model_dump_json(indent=2))
print()
print(f"Type: {type(nanny).__name__}; certifications field is now a {type(nanny.certifications).__name__}")

In [ ]:
# 3d. Same pattern for the parent intake.
raw_json = client.complete(
    model="gpt-4o-mini",
    messages=[{
        "role": "user",
        "content": PARENT_INTAKE_EXTRACTION_PROMPT.replace("{intake_text}", parent_text),
    }],
    temperature=0.0,
    response_format={"type": "json_object"},
)
data = json.loads(raw_json)
data["id"] = "p_demo"

parent = ParentIntake.model_validate(data)
print(parent.model_dump_json(indent=2))

In [ ]:
# 3e. Side-by-side: what can each output type do for you?
print("FREE-FORM lets you display nice prose to a human.")
print("  - Length:", len(freeform), "chars of prose")
print()
print("SCHEMA-LOCKED lets you program against typed data.")
print(f"  - Years experience: {nanny.years_experience}")
print(f"  - CPR? {'CPR' in [c.upper().replace(' ', '') for c in nanny.certifications] or any('CPR' in c for c in nanny.certifications)}")
print(f"  - Available on weekends: {'sat' in nanny.availability_days or 'sun' in nanny.availability_days}")

In [ ]:
# 3f. Now run extraction across ALL 10 PDFs and write the result to a dict
# keyed by id. This is what powers the matching demo in section 4.
# (Note: data/seed_db.json already ships pre-baked from Plan 1 as a fallback.)

all_nannies = []
for i, pdf in enumerate(sorted(PDF_DIR.glob("nanny_resume_*.pdf")), start=1):
    text = extract_text(pdf)
    raw = client.complete(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": NANNY_RESUME_EXTRACTION_PROMPT.replace("{resume_text}", text)}],
        temperature=0.0,
        response_format={"type": "json_object"},
    )
    d = json.loads(raw)
    d["id"] = f"n_{i:02d}"
    all_nannies.append(NannyProfile.model_validate(d))

all_parents = []
for i, pdf in enumerate(sorted(PDF_DIR.glob("parent_intake_*.pdf")), start=1):
    text = extract_text(pdf)
    raw = client.complete(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": PARENT_INTAKE_EXTRACTION_PROMPT.replace("{intake_text}", text)}],
        temperature=0.0,
        response_format={"type": "json_object"},
    )
    d = json.loads(raw)
    d["id"] = f"p_{i:02d}"
    all_parents.append(ParentIntake.model_validate(d))

print(f"Extracted {len(all_nannies)} nannies and {len(all_parents)} parents.")
print("First nanny names:", [n.name for n in all_nannies[:3]])

In [ ]:
# 🎯 TRY IT:
#   - Tighten NannyProfile (in src/nanny_workshop/models.py) to require
#     years_experience >= 1, then re-run 3c on a resume of a brand-new nanny.
#     What happens? (Spoiler: Pydantic raises a clear error rather than silently
#     accepting bad data.)
#   - Change the prompt in nanny_workshop/prompts.py to ask the model to also
#     extract "hourly_rate_usd" — note that without adding the field to the
#     Pydantic model, the model's invented hourly_rate is dropped at validate time.
#
# Type-locked outputs let you change the contract in one place and have callers fail loudly.

## 4. Embeddings + matching with ChromaDB

Now that we have typed records, we can compute embeddings — fixed-size vectors that put semantically similar text near each other in vector space. Then a parent intake is just a search query in that space.

We'll:
1. Compose a search-friendly "document" for each nanny and parent
2. Embed each document with `text-embedding-3-small`
3. Store everything in **ChromaDB** (`nanny_db/`)
4. Query: given a parent intake, return the top-3 most relevant nannies
5. **Visualize** all profiles in 2D with UMAP, color-coded by role

In [ ]:
# 4a. Compose a single text "document" per profile for embedding.
# The choice of WHAT to include in this document is a design decision —
# it determines what the embeddings can capture. Try removing fields later.

def nanny_to_doc(n) -> str:
    return (
        f"Nanny {n.name}. "
        f"{n.years_experience} years experience. "
        f"Certifications: {', '.join(n.certifications) or 'none'}. "
        f"Languages: {', '.join(n.languages) or 'English'}. "
        f"Available: {', '.join(n.availability_days) or 'flexible'}. "
        f"Pet-friendly: {n.pet_friendly}. "
        f"Special skills: {', '.join(n.special_skills) or 'general childcare'}. "
        f"{n.bio}"
    )

def parent_to_doc(p) -> str:
    kids = "; ".join(
        f"{c.age_years}yo{' (naps)' if c.needs_nap else ''}" for c in p.children
    )
    return (
        f"Family {p.family_name}. "
        f"Children: {kids}. "
        f"Needs care: {', '.join(p.schedule_days) or 'flexible'} "
        f"({p.schedule_hours_per_day}h/day). "
        f"Must have: {', '.join(p.must_haves) or 'none'}. "
        f"Nice to have: {', '.join(p.nice_to_haves) or 'none'}. "
        f"Neighborhood: {p.neighborhood}. "
        f"Notes: {p.notes}"
    )

print(nanny_to_doc(all_nannies[0]))
print()
print(parent_to_doc(all_parents[0]))

In [ ]:
# 4b. Embed all profiles and add them to a persistent Chroma collection.
# The collection lives in `nanny_db/` at repo root — Notebook 2 will read from there.

from nanny_workshop.chroma_client import NannyChroma

db = NannyChroma(persist_dir=ROOT / "nanny_db", collection_name="profiles")

# Only add records that aren't already in the collection (rerun-safe).
existing = db.count()
if existing == 0:
    nanny_docs = [nanny_to_doc(n) for n in all_nannies]
    parent_docs = [parent_to_doc(p) for p in all_parents]

    nanny_vecs = [client.embed(model="text-embedding-3-small", text=d) for d in nanny_docs]
    parent_vecs = [client.embed(model="text-embedding-3-small", text=d) for d in parent_docs]

    db.add(
        ids=[n.id for n in all_nannies],
        documents=nanny_docs,
        metadatas=[{"role": "nanny", "name": n.name} for n in all_nannies],
        embeddings=nanny_vecs,
    )
    db.add(
        ids=[p.id for p in all_parents],
        documents=parent_docs,
        metadatas=[{"role": "parent", "family": p.family_name} for p in all_parents],
        embeddings=parent_vecs,
    )

print(f"Chroma collection now has {db.count()} records (was {existing}).")

In [ ]:
# 4c. Given a parent intake, return the top-3 nannies — searching only the
# "nanny" subset using a metadata filter (where={"role": "nanny"}).

target_parent = all_parents[0]
target_doc = parent_to_doc(target_parent)
target_vec = client.embed(model="text-embedding-3-small", text=target_doc)

results = db.query(
    query_embedding=target_vec,
    n_results=3,
    where={"role": "nanny"},
)

print(f"Top 3 nanny matches for family {target_parent.family_name}:")
for rank, (nanny_id, meta, distance) in enumerate(
    zip(results["ids"][0], results["metadatas"][0], results["distances"][0]),
    start=1,
):
    print(f"  {rank}. {meta['name']} (id={nanny_id}) — distance {distance:.3f}")

In [ ]:
# 4d. Project all profile embeddings into 2D using UMAP and plot them.
# Nannies and parents should form (roughly) distinct clusters if their
# documents differ in vocabulary — which they do here.

import numpy as np
import matplotlib.pyplot as plt
import umap

# Re-fetch the embeddings from Chroma so the plot reflects what's stored.
all_records = db._collection.get(include=["embeddings", "metadatas"])
embeddings = np.array(all_records["embeddings"])
roles = [m["role"] for m in all_records["metadatas"]]

reducer = umap.UMAP(n_neighbors=5, min_dist=0.3, random_state=42)
coords = reducer.fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(8, 6))
for role, color in [("nanny", "tab:blue"), ("parent", "tab:orange")]:
    mask = np.array([r == role for r in roles])
    ax.scatter(coords[mask, 0], coords[mask, 1], c=color, label=role, s=120, alpha=0.8)

for i, (m, (x, y)) in enumerate(zip(all_records["metadatas"], coords)):
    label = m.get("name") or m.get("family") or all_records["ids"][i]
    ax.annotate(label, (x, y), fontsize=8, alpha=0.7)

ax.set_title("Nanny & parent profiles in embedding space (UMAP 2D)")
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 4e. WARNING — embedding distance is not the same as a good MATCH.
# Two profiles can be lexically similar but miss a hard constraint.
# Let's check whether our top-1 match actually meets the parent's must-haves.

top_nanny_id = results["ids"][0][0]
top_nanny = next(n for n in all_nannies if n.id == top_nanny_id)

print(f"Parent must-haves: {target_parent.must_haves}")
print(f"Top-1 nanny certifications: {top_nanny.certifications}")
print(f"Top-1 nanny pet-friendly: {top_nanny.pet_friendly}")
print()
print("This is the point of Notebook 2 — an agent that REASONS about constraints,")
print("not just one that ranks by cosine similarity.")

In [ ]:
# 🎯 TRY IT:
#   - Remove fields from nanny_to_doc (e.g., drop `certifications`) and re-run
#     embed + match. Watch the top-3 order shift.
#   - Change the `where={"role": "nanny"}` filter to `where={"role": "parent"}` —
#     now the "match" returns OTHER parents nearby in the space, which can be useful
#     for discovering similar family situations.
#   - Tighten the UMAP `n_neighbors` and see the clusters change shape.